# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and analyze the FAIR² colorectal cancer survivors dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset's structure and metadata are described by a Croissant schema hosted online at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed. If not, install it.
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata summary
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets and field `@id`s. In Croissant, every entity (record set, field, column) is uniquely identified by an `@id`. Let's inspect the record sets and their fields using the `mlcroissant` API.

In [ ]:
# List all record sets (@id) defined in the dataset.
print("Available record sets (by @id):")
for rs in dataset.record_sets:
    print(f"  - {rs['@id']}: {rs.get('name','')}")

# For demonstration, list fields for each record set
for rs in dataset.record_sets:
    print(f"\nFields in record set {rs['@id']}:")
    for field in rs.get('field', []):
        # Field can be either dict or @id string, resolve as dict
        field_dict = field if isinstance(field, dict) else dataset._find_by_id(field)
        print(f"  - {field_dict['@id']}: {field_dict.get('name','')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for exploratory analysis.

_Note: All entities are referenced exclusively by their Croissant `@id`. If you wish to use a specific field/column later, use the correct `@id` string._

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns for record set: {record_set_id}")
    print(f"  Columns: {list(df.columns)}\n")

# Choose one main record set for further analysis (here, the first available)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns in main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps: filtering, normalization, and grouping. We'll use only entity `@id`s to reference the relevant fields.

In [ ]:
# For demonstration, try to find a numeric field within the main record set
main_df = dataframes[main_record_set_id]

# Try to automatically guess a numeric field by dtype or name (@id)
numeric_field_id = None
for col in main_df.columns:
    # Choose if dtype numeric or id has 'age', 'interval', or 'count', fallback otherwise
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break
    elif any(word in col.lower() for word in ['age','interval','count','duration']):
        numeric_field_id = col
        break
if numeric_field_id is None and len(main_df.columns)>0:
    numeric_field_id = main_df.columns[0]

print(f"Selected numeric field (@id): {numeric_field_id}")

try:
    # Convert column to numeric if possible, errors set to NaN
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = main_df[numeric_field_id].mean() # Use mean as an example threshold
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to find groupable categorical field: look for 'sex', 'group', or non-numeric columns
    group_field_id = None
    for col in main_df.columns:
        if col == numeric_field_id:
            continue
        if any(word in col.lower() for word in ['sex','group','site','anatomical','location','status','type']):
            group_field_id = col
            break
        if not pd.api.types.is_numeric_dtype(main_df[col]):
            group_field_id = col
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id} per group):")
        display(grouped_df)
    else:
        print("No appropriate group field detected for grouping.")

except Exception as e:
    print(f"Could not perform numeric EDA due to: {e}")

## 5. Visualization
Visualize a key distribution or relationship using field `@id`. All axes must use the `@id` of each field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the main numeric field using @id as the label
if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Scatter plot numeric field vs. group field if available
if group_field_id and group_field_id in main_df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated step-by-step how to:
- Load a Croissant-encoded biomedical dataset using `mlcroissant`.
- Examine the record sets and fields via their `@id`s.
- Extract the tabular data for analysis.
- Perform basic EDA: filtering, normalization, grouping, and visualization entirely with Croissant `@id` references.

This workflow is reproducible and FAIR-compliant, ensuring full traceability of data elements via `@id`.